# Wavelet subsample — uniform per session (paw, left paw, wheel, left paw + wheel)

Copy of `3.2_wavelet_subsample.ipynb` with the training-set construction changed. The
original is left untouched.

**What changed**

| | `3.2_wavelet_subsample` | this notebook |
|---|---|---|
| frames kept per session | 2,000 of 20,000, drawn with p ∝ KDE density of a per-session t-SNE | 2,000 **uniformly at random** from all valid frames |
| features | 32 columns (raw positions + 0.5–32 Hz) for the t-SNE; 3.3 then clusters on 20 | only the 20 columns 3.3 clusters on |
| normalisation | subsample z-scored by its **own** stats | nothing applied; the **full-session** stats (raw and log) are saved so 3.3 applies exactly what it later applies to every frame |

**Why not the density weighting.** Berman et al. 2014 (*J R Soc Interface*, "Mapping the
stereotyped behaviour of freely moving fruit flies") run t-SNE on 20,000 random points per
fly, watershed the density, and draw from each region **in proportion to the integral of the
density over that region**: stratified sampling that *preserves* each individual's
composition while making sure every region is represented. Drawing individual points with
p ∝ density instead weights them by density twice (they already occur in proportion to it).
On 25 sessions that made the training set's cluster make-up differ from the session's by a
total variation of 0.09, against 0.02 for a uniform draw, with the rarest vigorous cluster
at ×0.71. A uniform draw is Berman's scheme in expectation, without the t-SNE.

Every session still contributes the same number of frames, so a long session does not
outvote a short one.

**What it can subsample** (`DATASET`, `VAR`; definitions in the first code cell):
- `DATASET`: `'proficient'` (`data/`), `'neuromodulators'` (`data/neuromodulators/`),
  or `'training'` (`data/training/`).
- `VAR`: `'paw'` (both paws, 20 features), `'left_paw'` (10), `'wheel'` (5), or
  `'left_paw_wheel'` (15: left paw and wheel wavelets merged on `Bin`). This replaces
  `3.2_paw_wheel_subsample.ipynb`.

Wheel uses 0.5–8 Hz like the paw. The original 3.3's wheel-only setting used 4–32 Hz.

In [1]:
"""SHARED DEFINITIONS -- identical in 3.2_..._uniform and 3.3_..._uniform."""
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA = '/home/ines/repositories/representation_learning_variability/paper-individuality/data/'

# each dataset keeps its wavelet folders under its own root
DATASETS = {'proficient': '', 'neuromodulators': 'neuromodulators/', 'training': 'training/'}

BANDS = ['0.5', '1.0', '2.0', '4.0', '8.0']     # the bands the clustering uses
# source -> (wavelet folders to try in order, file prefix, columns, log floor)
#   log floor: log(x + floor), only used with LOG = True. Paw amplitudes have a median of ~1-6
#   and a 1st percentile of ~0.01-0.07, so 0.1 compresses a still paw's tracking jitter. Wheel
#   amplitudes are ~20x smaller (still ~1e-4, moving ~0.05-0.3), so its floor is 0.005.
#   16 / 32 Hz are left out everywhere: for the wheel they are encoder noise.
SOURCES = {
    'paw':      (['paw_wavelets/'], 'paw_vel_wavelets_',
                 [f'{p}_{a}{b}' for p in ('l_paw', 'r_paw') for a in 'xy' for b in BANDS], 0.1),
    # the proficient dataset keeps the left paw inside the both-paw files
    'left_paw': (['left_paw_wavelets/', 'paw_wavelets/'], 'paw_vel_wavelets_',
                 [f'l_paw_{a}{b}' for a in 'xy' for b in BANDS], 0.1),
    'wheel':    (['wheel_wavelets/'], 'wheel_vel_wavelets_',
                 [f'avg_wheel_vel{b}' for b in BANDS], 0.005),
}
# what can be clustered: one source, or several merged on Bin
VARS = {'paw': ['paw'], 'left_paw': ['left_paw'], 'wheel': ['wheel'],
        'left_paw_wheel': ['left_paw', 'wheel']}


def features(var):
    return [c for s in VARS[var] for c in SOURCES[s][2]]


def feature_sources(var):
    """The source of every feature column, in features() order."""
    return [s for s in VARS[var] for _ in SOURCES[s][2]]


def feature_eps(var):
    return np.array([SOURCES[s][3] for s in feature_sources(var)])


def source_dir(dataset, src):
    root = DATA + DATASETS[dataset]
    for d in SOURCES[src][0]:
        if os.path.isdir(root + d):
            return root + d
    raise FileNotFoundError(f'no {SOURCES[src][0]} under {root}')


def session_tags(dataset, var):
    """'<eid>_<mouse>' of every session that has a file for EVERY source of `var`."""
    have = []
    for s in VARS[var]:
        d, prefix = source_dir(dataset, s), SOURCES[s][1]
        have.append({f[len(prefix):] for f in os.listdir(d) if f.startswith(prefix)})
    return sorted(set.intersection(*have))


def load_frame(dataset, var, tag, extra=()):
    """One session, its sources merged on Bin -> Bin, the features, and whichever `extra`
    columns (raw traces for plotting) the files have."""
    out = None
    for s in VARS[var]:
        path = source_dir(dataset, s) + SOURCES[s][1] + tag
        names = set(pq.read_schema(path).names)
        taken = set() if out is None else set(out.columns)
        cols = ['Bin'] + SOURCES[s][2] + [c for c in extra if c in names and c not in taken]
        df = pd.read_parquet(path, columns=cols)
        out = df if out is None else out.merge(df, on='Bin', how='inner')
    return out


def load_session(dataset, var, tag):
    """-> (raw amplitudes of the frames with every feature finite, their Bin, the mask)."""
    df = load_frame(dataset, var, tag)
    X = df[features(var)].to_numpy(float)
    valid = np.isfinite(X).all(axis=1)
    return X[valid], df['Bin'].to_numpy()[valid], valid


# TRAINING STAGES: each is a subfolder of design_matrices/1_camera_setup/ listing its sessions
STAGE_DIR = DATA + 'design_matrices/1_camera_setup/'


def stage_eids(stage):
    """The session eids of one training stage, e.g. 'session_1', 'last_training', 'biased'."""
    return {f.split('_')[2] for f in os.listdir(STAGE_DIR + stage) if 'design_matrix' in f}


def dataset_root(dataset, out_root=None):
    """Where a dataset's subsamples and states live: its own folder under DATA, or the same
    layout under `out_root` (test runs)."""
    return (out_root or DATA) + DATASETS[dataset]


In [4]:
"""KNOBS"""
DATASET = 'proficient'  # 'proficient' | 'neuromodulators' | 'training'
VAR     = 'left_paw_wheel'         # 'paw' | 'left_paw' | 'wheel' | 'left_paw_wheel'
N_SUB   = 2000          # frames per session -- every session weighs the same
SEED    = 0
MAX_SESSIONS = None     # an int runs only the first n sessions, for a quick test
OUT_ROOT     = None     # None = DATA; a path redirects the output (tests), same layout

OUT = dataset_root(DATASET, OUT_ROOT) + f'{VAR}_subsampled_wavelets_uniform/'
os.makedirs(OUT, exist_ok=True)
tags = session_tags(DATASET, VAR)[:MAX_SESSIONS]
EPS = feature_eps(VAR)          # the log floor of each feature's source (see SOURCES)
print(f'{DATASET}, {VAR}: {len(tags)} sessions, {len(features(VAR))} features -> {OUT}')

proficient, left_paw_wheel: 332 sessions, 15 features -> /home/ines/repositories/representation_learning_variability/paper-individuality/data/left_paw_wheel_subsampled_wavelets_uniform/


In [5]:
"""SUBSAMPLE -- one .npz per session."""
rng = np.random.default_rng(SEED)
too_short = []
for tag in tags:
    X, bins, _ = load_session(DATASET, VAR, tag)
    if len(X) < N_SUB:
        too_short.append((tag, len(X)))
        continue
    pick = np.sort(rng.choice(len(X), N_SUB, replace=False))
    L = np.log(X + EPS)
    np.savez(OUT + tag, sub=X[pick], bins=bins[pick], features=np.array(features(VAR)),
             n_valid=len(X), eps=EPS,
             mean_raw=X.mean(0), sd_raw=X.std(0),       # ddof=0, as scipy.stats.zscore
             mean_log=L.mean(0), sd_log=L.std(0))
print(f'saved {len(tags) - len(too_short)}; too short: {too_short}')

saved 332; too short: []
